In [1]:
#imports
from bertopic import BERTopic
from gensim.models import Phrases
from gensim.models.phrases import Phraser
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import spacy
import pandas as pd, re, numpy as np, torch
from transformers import AutoTokenizer, AutoModel


In [2]:
# 0) remove custom tokens
def remove_words(text, words_to_remove, min_length=3):
    escaped = [re.escape(w) for w in words_to_remove]
    pat = r'(?<!\S)(?:' + '|'.join(escaped) + r')(?!\S)'
    cleaned = re.sub(pat, '', text, flags=re.IGNORECASE)
    cleaned = ' '.join(cleaned.split())
    return ' '.join(tok for tok in cleaned.split() if len(tok) >= min_length)

In [3]:
# 1)load & filter
df = pd.read_csv('final_vg_all.csv', index_col=0)
df['processed'] = df['processed'].astype(str)
df = df.drop(['topic', 'probability', 'label', 'meta'], axis=1)
df.head()


,title,outlet,date,authors,body,word_count,processed,climate_related,year,PERSON,...,GPE,General,Migrants_Refugees,Children,Elderly,Poor,Other_Vulnerable,Gender_and_Sexuality,Disabled,References_Vulnerable_Groups
0,Arcadis pakt stormvloedkeringen aan: droge voe...,Telegraaf,2024-01-04,Yteke de Jong,Ingenieursbureau Arcadis pakt de komende jaren...,587,ingenieursbureau jaar onderhoud stormvloedkeri...,yes,2024,"['Ingenieursbureau Arcadis', 'Wolter', 'Arcadi...",...,"['Amsterdam', 'Nederland', 'Duitsland', 'Limbu...",False,False,False,False,False,False,False,False,False
1,Biden gaat naar Egypte voor klimaatconferentie,Telegraaf,2022-10-29,Unknown Authors,De Amerikaanse president Joe Biden gaat volgen...,263,president cop27klimaattop huis democraat noodz...,yes,2022,"['Joe Biden', 'Vladimir', 'kroonprins Mohammed...",...,"['Verenigde Naties', 'Egypte', 'Sharm', 'Cambo...",False,False,False,False,False,False,False,False,False
2,Omstreden wetenschapper Steven Koonin waarschu...,Telegraaf,2023-09-30,Kleis Jager,De natuurkundige Steven Koonin adviseerde Bara...,856,steven barack_obama wereld boodschap klimaatve...,yes,2023,"['Steven Koonin', 'Barack Obama', 'Koonin', 'U...",...,"['Nederland', 'Verenigde Staten', 'Europa', 'VS']",False,False,False,False,False,False,False,False,False
3,’Weerzin tegen windmolen’,Telegraaf,2020-12-11,Unknown Authors,Johan Sijtsema vindt het opvallend dat de Twee...,102,windmolen gevolg lichtvervuiling parlement hor...,yes,2020,['Johan Sijtsema'],...,[],False,False,False,False,False,False,False,False,False
4,Honger verslindt levens: 1 op 5 jonge kinderen...,Telegraaf,2023-12-15,Unknown Authors,"Kinderen horen te spelen, rennen en lachen. Ma...",675,kind waarheid kind levensjaar ondervoeding dre...,yes,2023,"['Sahra', 'Annegré de Roos']",...,"['Burkina', 'Faso', 'Mali', 'Somalië']",False,False,True,False,False,False,False,False,True


In [373]:
df['outlet'].value_counts()

outlet
AD            3637
Trouw         3406
Volkskrant    2770
FD            2023
NRC           1691
Telegraaf     1125
Parool        1094
Name: count, dtype: int64

In [4]:
# 2) bigrams + remove custom stop-words
sentences = [doc.split() for doc in df['processed']]
bigram = Phrases(sentences, min_count=10, threshold=150)
bigram_ph = Phraser(bigram)

custom_stopwords = [
    'r','uur','pagina_pagina','dag','tijd','maand','aantal',
    'week','vraag','br/_>','programma_film','<_br/','moment',
    'keer','deel','beetje','soort','lexisnexis_lexisnexis','facebookpagina','date','load'
]

df['processed'] = (
    df['processed']
      .apply(lambda txt: " ".join(bigram_ph[txt.split()]))
      .apply(lambda txt: remove_words(txt, custom_stopwords, min_length=3))
)
df = df[df['processed'].str.strip() != '']
texts = df['processed'].tolist()


In [5]:
# 3) Dutch stop-words & vectorizer
nlp = spacy.load('nl_core_news_lg')
dutch_stops = list(nlp.Defaults.stop_words)
vectorizer = CountVectorizer(
    stop_words=dutch_stops,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.6
)


In [ ]:
import random
seeds = random.sample(range(1, 10_000), 20)

In [182]:
seeds

[504, 8170, 7682, 8397, 6903, 3962, 90, 9177, 7594, 205]

In [356]:
# 4) UMAP & HDBSCAN for tighter clusters
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.1,
    random_state=205
)
hdbscan_model = HDBSCAN(
    min_cluster_size=5,
    min_samples=8,
    cluster_selection_epsilon=0.1,
    cluster_selection_method='eom',
    metric='euclidean',
    prediction_data=True
)

In [357]:
# 5) Dutch BERT embedding helper
tokenizer = AutoTokenizer.from_pretrained('GroNLP/bert-base-dutch-cased')
hf_model = AutoModel.from_pretrained('GroNLP/bert-base-dutch-cased')
hf_model.eval()

def embed_texts(texts, batch_size=32):
    embs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, return_tensors='pt')
            out = hf_model(**enc).last_hidden_state
            mask = enc.attention_mask.unsqueeze(-1).float()
            summed = (out * mask).sum(1)
            counts = mask.sum(1).clamp(min=1e-9)
            embs.append((summed / counts).cpu().numpy())
    return np.vstack(embs)

Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [358]:
# 6) precompute document embeddings
embeddings = embed_texts(texts, batch_size=16)

In [359]:
# 7) wrapper so BERTopic uses 768-dim BERT embeddings
class HFEmbedder:
    def embed_documents(self, docs):
        return embed_texts(docs)
    def embed_queries(self, docs):
        return embed_texts(docs)

In [360]:
# 8) Instantiate BERTopic with custom components
topic_model = BERTopic(
    embedding_model=HFEmbedder(),
    vectorizer_model=vectorizer,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True
)

In [361]:
# 9) Fit & transform
topics, probs = topic_model.fit_transform(texts)

2025-10-21 11:59:07,164 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/493 [00:00<?, ?it/s]

2025-10-21 11:59:51,131 - BERTopic - Embedding - Completed ✓
2025-10-21 11:59:51,131 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-10-21 11:59:59,662 - BERTopic - Dimensionality - Completed ✓
2025-10-21 11:59:59,663 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-10-21 12:00:23,509 - BERTopic - Cluster - Completed ✓
2025-10-21 12:00:23,511 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-10-21 12:00:26,912 - BERTopic - Representation - Completed ✓


In [362]:
# 10) reduce outliers
new_topics = topic_model.reduce_outliers(
    documents=texts,
    topics=topics,
    threshold=0.25,
    strategy='embeddings'
)

# 11) update representations (optional)
topic_model.update_topics(texts, new_topics)

# — **insert this** —
topic_model.topics_ = new_topics

# 12) now get_topic_info() will reflect your remapped labels
topic_info = topic_model.get_topic_info()

In [363]:
# 13) reduce outliers without manual embeddings
new_topics = topic_model.reduce_outliers(
    documents=texts,
    topics=topics,
    threshold=0.25,
    strategy='embeddings'
)

In [364]:
# 14) update model to reflect new topics
# this recalculates topic representations; probabilities remain from fit_transform
topic_model.update_topics(
    texts,
    new_topics
)

In [365]:
from collections import Counter

# … after reduce_outliers() and update_topics() …

# 1) Overwrite the stored labels and docs
topic_model.topics_    = new_topics
topic_model.documents_ = texts

# 2) Recompute the size‐mapping from your new labels
topic_model.topic_sizes_ = dict(Counter(new_topics))

# 3) Now get_topic_info() will match your df
topic_info = topic_model.get_topic_info()

# 4) Finally, add your top‐30 representation column as before
NEW_TOP_N_WORDS = 30
reprs = []
for tid in topic_info.Topic:
    if tid == -1:
        reprs.append("Outlier Topic / Not Applicable")
    else:
        words_scores = topic_model.get_topic(tid)
        reprs.append(", ".join(w for w, _ in words_scores[:NEW_TOP_N_WORDS]))
topic_info[f"Representation_Top_{NEW_TOP_N_WORDS}"] = reprs

with pd.option_context('display.max_colwidth', None):
    print(topic_info[['Topic','Count','Name',f"Representation_Top_{NEW_TOP_N_WORDS}"]].head(50))


    Topic  Count                                                        Name  \
0      -1     25                                                         NaN   
1       0    364                                   0_bank_ecb_ing_instelling   
2       1    311                         1_graad_hitte_temperatuur_hittegolf   
3       2    252                           2_brand_bosbrand_vuur_natuurbrand   
4       3    223                         3_virus_coronavirus_pandemie_vaccin   
5       4    237                                     4_insect_plant_tuin_bij   
6       5    232                5_windmolen_windpark_windturbine_windenergie   
7       6    419                            6_activist_protest_actie_politie   
8       7    284                        7_minister_kabinet_ministerie_partij   
9       8    175                     8_film_documentaire_regisseur_personage   
10      9    366                          9_plant_biodiversiteit_natuur_dier   
11     10    133                10_kerne

In [366]:
# 12) Assign & report) Assign & report) Assign & report
df['topic'] = new_topics
df['probability'] = [probs[i, t] if t >= 0 else 0.0 for i, t in enumerate(new_topics)]
print(f"Final noise rate: {(df['topic'] == -1).mean():.2%}")

Final noise rate: 0.16%


In [367]:
# 1) Lengths match?
assert len(new_topics) == len(df), (
    f"Mismatch: {len(new_topics)} topics vs. {len(df)} rows"
)
assert probs.shape[0] == len(df), (
    f"Mismatch: {probs.shape[0]} probability rows vs. {len(df)} rows"
)

# 2) Spot‐check a handful of rows:
for idx in [0, 5, 50, 100, len(df)-1]:
    assigned_t = df.at[idx, 'topic']
    assigned_p = df.at[idx, 'probability']
    recomputed_p = probs[idx, assigned_t] if assigned_t >= 0 else 0.0
    print(idx, assigned_t, assigned_p, recomputed_p)
    assert abs(assigned_p - recomputed_p) < 1e-8, (
        f"Row {idx}: stored {assigned_p} vs. recomputed {recomputed_p}"
    )

print("✅ All lengths and spot‐checks line up.")


0 75 0.002248870720081307 0.002248870720081307
5 13 0.0037242752372464365 0.0037242752372464365
50 75 0.0004087467008669066 0.0004087467008669066
100 50 0.01474420144769188 0.01474420144769188
15745 9 0.018104504956989696 0.018104504956989696
✅ All lengths and spot‐checks line up.


In [368]:
df

,title,outlet,date,authors,body,word_count,processed,climate_related,year,PERSON,...,Migrants_Refugees,Children,Elderly,Poor,Other_Vulnerable,Gender_and_Sexuality,Disabled,References_Vulnerable_Groups,topic,probability
0,Arcadis pakt stormvloedkeringen aan: droge voe...,Telegraaf,2024-01-04,Yteke de Jong,Ingenieursbureau Arcadis pakt de komende jaren...,587,ingenieursbureau jaar onderhoud stormvloedkeri...,yes,2024,"['Ingenieursbureau Arcadis', 'Wolter', 'Arcadi...",...,False,False,False,False,False,False,False,False,75,0.002249
1,Biden gaat naar Egypte voor klimaatconferentie,Telegraaf,2022-10-29,Unknown Authors,De Amerikaanse president Joe Biden gaat volgen...,263,president cop27klimaattop huis democraat noodz...,yes,2022,"['Joe Biden', 'Vladimir', 'kroonprins Mohammed...",...,False,False,False,False,False,False,False,False,113,0.053110
2,Omstreden wetenschapper Steven Koonin waarschu...,Telegraaf,2023-09-30,Kleis Jager,De natuurkundige Steven Koonin adviseerde Bara...,856,steven barack_obama wereld boodschap klimaatve...,yes,2023,"['Steven Koonin', 'Barack Obama', 'Koonin', 'U...",...,False,False,False,False,False,False,False,False,11,0.050738
3,’Weerzin tegen windmolen’,Telegraaf,2020-12-11,Unknown Authors,Johan Sijtsema vindt het opvallend dat de Twee...,102,windmolen gevolg lichtvervuiling parlement hor...,yes,2020,['Johan Sijtsema'],...,False,False,False,False,False,False,False,False,5,0.353933
4,Honger verslindt levens: 1 op 5 jonge kinderen...,Telegraaf,2023-12-15,Unknown Authors,"Kinderen horen te spelen, rennen en lachen. Ma...",675,kind waarheid kind levensjaar ondervoeding dre...,yes,2023,"['Sahra', 'Annegré de Roos']",...,False,True,False,False,False,False,False,True,25,0.201673
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15741,'Groentezaden zijn kostbaarder dan goud' 'Gro...,Parool,2015-05-18,HANNEKE KEULTJES,Myanmar: Nederland verkoopt landbouwproducten ...,823,Landbouwproducten training dictatuur geld slag...,yes,2015,"['Sharon Dijksma', 'Aung', 'Kyi', 'De Heus', '...",...,False,False,False,False,False,False,False,False,153,0.065919
15742,"'Tegeltje eruit, plantje erin' 'Tegeltje eruit...",Parool,2015-11-02,MARCEL WIEGMAN,Water Week: Waar moet al dat water heen? \nDe ...,614,water water kassei dam gras aandacht klimaatve...,yes,2015,"['Verbaasde', 'Udo Kock van Financiën', 'Anne ...",...,False,False,False,False,False,False,False,False,13,0.010525
15743,'Klimaatverandering is risicoverdubbelaar' 'Kl...,Parool,2016-12-06,HANNEKE KEULTJES,Oorlog: Opwarming aarde onderliggende oorzaak ...,697,oorlog opwarming aarde oorzaak strijd generaal...,yes,2016,"['Tom Middendorp', 'Middendorp', 'Assad', 'een...",...,False,False,False,False,False,False,False,False,120,0.065953
15744,Milieu hoger op de agenda bij Shell Milieu hog...,Parool,2015-05-18,HENK SCHUTTEN,Oliewinning: Proefboringen in Noordpoolgebied ...,545,oliewinning proefboring noordpoolgebied golf p...,yes,2015,"['Olievrije', 'Discoverer', 'Ed Murray', 'Matt...",...,False,False,False,False,False,False,False,False,26,0.063768


In [369]:
total_docs = len(df)
noise_docs = (df['topic'] == -1).sum()
print(f"DF:   total={total_docs:,}, noise={noise_docs:,}, frac={noise_docs/total_docs:.2%}")


DF:   total=15,746, noise=25, frac=0.16%


In [370]:
df.to_csv('CC_bert_205.csv')

In [371]:
topic_model.save("bertopic_model_CC_205")


2025-10-21 12:01:22,910 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [372]:
topic_info.to_excel("topics_overview_CC_205.xlsx", index=False)
